# [B2/RQ1] Do we have the public data to run the STTR spinout-vs-subcontract cascade?

**Status:** exploratory
**Research question:** B2 — STTR spinout vs. subcontract relationship, dedicated inventory
entry (RQ1) ([`specs/sttr-spinout-linkage/design.md`](../../specs/sttr-spinout-linkage/design.md));
**not** `docs/research-questions.md`'s pre-existing B2 (award-to-contract transition) — the spec
notes this anchor collision explicitly
([design.md anchor verification](../../specs/sttr-spinout-linkage/design.md#anchor-verification-and-a-correction-to-the-brief)).
**Decision this informs:** whether task 1.3 (implement the RQ1 cascade) is blocked on missing
public-data sources, on the unresolved `open-questions.md` gate, or both — before any
implementation effort is spent.
**Data as of:** whatever local artifacts are present (see the per-dimension checks below)
**Owner:** STTR spinout-linkage workbench (PR #615)

**This notebook does not implement or run the classification cascade.** `design.md` line 5-6 is
explicit that the design "does not authorize implementation, materialization, a headline cell, or
any citable claim" until the owner resolves `open-questions.md`; the PR body for #615 states the
RQ1 classifier is "Not run in this PR." This notebook only checks whether the five evidence
dimensions (D1-D5) the cascade would consume, and the six partner-type seed lists, exist locally —
a pure input-availability probe, same shape as the `input-check` cell in
[`b1_sttr_partner_type_commercialization.ipynb`](b1_sttr_partner_type_commercialization.ipynb).
No `SPINOUT_T1` / `SPINOUT_T2` / `SUBCONTRACT` / `INDETERMINATE` label is computed anywhere below.


## Data contract

- **Population:** none scored here. If it were run, the cascade's population would be SBIR.gov
  awards with `program = STTR` (D1 spine), evaluated per Phase II award.
- **What this notebook measures instead:** local artifact existence and, where an artifact exists,
  basic shape/coverage (row counts, non-null shares) for each of the five evidence dimensions in
  [`design.md`](../../specs/sttr-spinout-linkage/design.md#evidence-dimensions) (D1 award spine, D2
  person trail, D3 IP trail, D4 money/paper trail, D5 text trail), plus the six partner-type seed
  lists in [`seed-list-provenance.md`](../../specs/sttr-spinout-linkage/seed-list-provenance.md).
- **Keys:** none joined. Each dimension is checked independently; no cross-dimension join is
  attempted (that join is exactly the cascade this notebook does not run).
- **Missingness:** an artifact being absent means that dimension has no local source today, not
  that the dimension is unmeasurable in principle — D2/D4 in particular have live-API or
  bulk-download paths that are not yet materialized as local files.
- **Outputs:** an input-readiness table per dimension. Exploratory only. No canonical generator;
  no output is a scored `CANDIDATE` assertion.


In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd


def find_repo_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "sbir_etl").exists():
            return candidate
    raise RuntimeError("Run this notebook from inside the sbir-analytics checkout")


REPO_ROOT = find_repo_root()
REPO_ROOT

In [ ]:
AS_OF_DATE = "local-artifacts"  # this is an availability probe, not a frozen data cut

AWARD_CANDIDATES = (
    REPO_ROOT / "data" / "processed" / "enriched_sbir_awards.parquet",
    REPO_ROOT / "data" / "raw" / "sbir" / "award_data.csv",
)
DIMENSION_INPUTS: dict[str, dict[str, Path]] = {
    "D1_award_spine": {
        "awards": next((p for p in AWARD_CANDIDATES if p.exists()), AWARD_CANDIDATES[-1]),
    },
    "D2_person_trail": {
        # No local cache directory exists for OpenAlex/PubMed/ORCID in this repo today.
        # sbir_etl.enrichers.orcid_client queries ORCID live; no OpenAlex/PubMed fetcher exists.
        "orcid_cache": REPO_ROOT / "data" / "cache" / "orcid",
        "openalex_cache": REPO_ROOT / "data" / "cache" / "openalex",
    },
    "D3_ip_trail": {
        "uspto_bulk_assignment": REPO_ROOT / "data" / "raw" / "uspto" / "assignments" / "assignment.csv.zip",
        "uspto_bulk_assignor": REPO_ROOT / "data" / "raw" / "uspto" / "assignments" / "assignor.csv.zip",
        "uspto_bulk_assignee": REPO_ROOT / "data" / "raw" / "uspto" / "assignments" / "assignee.csv.zip",
        "uspto_bulk_conveyance": REPO_ROOT / "data" / "raw" / "uspto" / "assignments" / "assignment_conveyance.csv.zip",
        "uspto_transformed_dir": REPO_ROOT / "data" / "transformed" / "uspto",
    },
    "D4_money_paper_trail": {
        "form_d": REPO_ROOT / "data" / "form_d_details.jsonl",
        "usaspending_subaward_general": REPO_ROOT / "data" / "processed" / "usaspending" / "subawards.parquet",
    },
    "D5_text_trail": {
        # Same source as D1 — the SBIR.gov Abstract field. Checked separately for
        # column-level coverage, since D1 presence does not imply D5 usability.
        "awards": next((p for p in AWARD_CANDIDATES if p.exists()), AWARD_CANDIDATES[-1]),
    },
}
SEED_LIST_PROVENANCE = REPO_ROOT / "specs" / "sttr-spinout-linkage" / "seed-list-provenance.md"

## Sub-question 1 — does an artifact exist for each dimension's declared source?

Flat existence check across all five dimensions before looking at any single one in detail.

In [ ]:
rows = []
for dimension, sources in DIMENSION_INPUTS.items():
    for source_name, path in sources.items():
        rows.append(
            {
                "dimension": dimension,
                "source": source_name,
                "path": str(path.relative_to(REPO_ROOT)),
                "exists": path.exists(),
            }
        )
input_status = pd.DataFrame(rows)
input_status

## Honesty block (read before any table below)

1. **Existence is not usability.** A file existing does not mean it is joinable to the D1 spine,
   scoped to the STTR population, or free of test-fixture content — D3's `data/transformed/uspto/`
   files are checked for exactly this below.
2. **A missing D2/D4 local cache is not evidence those dimensions are unmeasurable.** D2 has a live
   ORCID API path with no local cache; D4's subaward marker has no *general* extract locally, only
   cohort-scoped extracts (e.g. nanotechnology) that do not cover the STTR population.
3. **Seed lists are a separate classifier's inputs.** The six partner-type seed lists
   (`ffrdc_master`, `ipeds_institutions`, `research_hospitals`, `new_model_orgs`,
   `fiscal_sponsors`, `nonprofit_registry`) feed the RI partner-type classifier, not the RQ1
   spinout/subcontract cascade — they are checked here because both classifiers share the D1 spine
   and the same freeze gate, not because they are the same measurement.
4. **This is a readiness probe, not a coverage estimate.** `coverage-memo.md` in the spec already
   gives per-dimension, per-agency expected-coverage estimates from the literature and pipeline
   design; this notebook does not repeat or supersede that analysis, it only checks what is
   physically present on disk today.


## D1 — award spine

Declared positive signal: SBC, RI, PI, agency, FY, abstract present (the join spine). Filter to
`program = STTR`, `phase = Phase II` and report non-null coverage on the fields the cascade's
Order-0 rule reads (`RI Name`, `PI Name`).

In [ ]:
def first_col(frame: pd.DataFrame, names: tuple[str, ...]) -> str | None:
    lookup = {c.lower(): c for c in frame.columns}
    for name in names:
        if name.lower() in lookup:
            return lookup[name.lower()]
    return None


awards_path = DIMENSION_INPUTS["D1_award_spine"]["awards"]
if not awards_path.exists():
    print(f"Missing {awards_path.relative_to(REPO_ROOT)} — D1 has no local source.")
    d1_coverage = pd.DataFrame()
else:
    awards_raw = (
        pd.read_parquet(awards_path)
        if awards_path.suffix == ".parquet"
        else pd.read_csv(awards_path, dtype=str, low_memory=False)
    )
    program_col = first_col(awards_raw, ("program", "Program"))
    phase_col = first_col(awards_raw, ("phase", "Phase"))
    ri_col = first_col(awards_raw, ("ri_name", "RI Name"))
    pi_col = first_col(awards_raw, ("pi_name", "PI Name"))
    is_sttr = awards_raw[program_col].fillna("").str.strip().str.upper().eq("STTR")
    is_p2 = (
        awards_raw[phase_col].fillna("").str.strip().str.upper().str.replace("PHASE ", "", regex=False)
        .isin({"II", "2"})
    )
    sttr_p2 = awards_raw.loc[is_sttr & is_p2]
    print(f"STTR Phase II award rows: {len(sttr_p2):,}")
    d1_coverage = pd.DataFrame(
        {
            "field": [ri_col, pi_col],
            "role": ["Order-0 D1-complete check", "Order-0 D1-complete check"],
            "non_null": [sttr_p2[ri_col].notna().sum(), sttr_p2[pi_col].notna().sum()],
            "total": [len(sttr_p2), len(sttr_p2)],
        }
    ).assign(coverage=lambda t: (t["non_null"] / t["total"]).round(4))
d1_coverage

## D2 — person trail

Declared source: OpenAlex / PubMed authorship, ORCID. Check for any locally cached corpus and for
which client code exists (live-API vs. cached).

In [ ]:
import importlib.util

orcid_cache = DIMENSION_INPUTS["D2_person_trail"]["orcid_cache"]
openalex_cache = DIMENSION_INPUTS["D2_person_trail"]["openalex_cache"]
print(f"ORCID local cache present: {orcid_cache.exists()} ({orcid_cache.relative_to(REPO_ROOT)})")
print(f"OpenAlex local cache present: {openalex_cache.exists()} ({openalex_cache.relative_to(REPO_ROOT)})")

has_orcid_client = importlib.util.find_spec("sbir_etl.enrichers.orcid_client") is not None
has_openalex_client = importlib.util.find_spec("sbir_etl.enrichers.openalex_client") is not None
print(f"sbir_etl.enrichers.orcid_client importable (live ORCID API client): {has_orcid_client}")
print(f"sbir_etl.enrichers.openalex_client importable: {has_openalex_client}")
print(
    "D2 status: ORCID has a live-API client and no local cache; "
    "OpenAlex/PubMed have neither a client nor a local cache in this repo."
)

## D3 — IP trail

Declared source: USPTO assignment data; Bayh-Dole government-interest statements. Two candidate
local sources exist at different processing stages — check both, and flag whether the
"transformed" snapshot is real bulk data or a test fixture.

In [ ]:
import json

bulk_paths = {
    k: v for k, v in DIMENSION_INPUTS["D3_ip_trail"].items() if k.startswith("uspto_bulk")
}
for name, path in bulk_paths.items():
    if path.exists():
        size_mb = path.stat().st_size / 1_000_000
        print(f"{name}: {size_mb:,.1f} MB present at {path.relative_to(REPO_ROOT)}")
    else:
        print(f"{name}: missing ({path.relative_to(REPO_ROOT)})")

transformed_dir = DIMENSION_INPUTS["D3_ip_trail"]["uspto_transformed_dir"]
if transformed_dir.exists():
    transformed_files = sorted(transformed_dir.glob("*.jsonl"))
    by_prefix: dict[str, list] = {}
    for f in transformed_files:
        prefix = f.stem.rsplit("_", 1)[0]
        by_prefix.setdefault(prefix, []).append(f)
    print(f"\n{len(transformed_files)} transformed jsonl files present, by kind:")
    for prefix, files in sorted(by_prefix.items()):
        total_kb = sum(f.stat().st_size for f in files) / 1_000
        print(f"  {prefix}: {len(files)} files, {total_kb:,.1f} KB total")

    sample_file = sorted(transformed_dir.glob("patent_assignments_*.jsonl"))[-1]
    with sample_file.open(encoding="utf-8") as handle:
        sample = json.loads(handle.readline())
    raw_text = sample_file.read_text(encoding="utf-8")
    looks_synthetic = "Widget" in raw_text and "Acme Corp" in raw_text
    print(f"\nSample record from {sample_file.name}: {sample}")
    print(
        f"Looks like a test fixture (contains placeholder names 'Widget'/'Acme Corp'): "
        f"{looks_synthetic} — if True, this is not a usable D3 corpus; only the bulk "
        ".csv.zip files under data/raw/uspto/assignments/ are real data, and are unprocessed."
    )
else:
    print(f"\nMissing {transformed_dir.relative_to(REPO_ROOT)}")


## D4 — money / paper trail

Declared source: USASpending subawards (subcontract marker); Form D officers/directors (spinout
marker, existing pipeline). Check both directions separately — they are scored independently in
the cascade.

In [ ]:
form_d_path = DIMENSION_INPUTS["D4_money_paper_trail"]["form_d"]
if form_d_path.exists():
    size_mb = form_d_path.stat().st_size / 1_000_000
    print(f"Form D (spinout-marker source): {size_mb:,.1f} MB present at {form_d_path.relative_to(REPO_ROOT)}")
else:
    print(f"Form D missing at {form_d_path.relative_to(REPO_ROOT)} — spinout-marker side of D4 not searched.")

subaward_general = DIMENSION_INPUTS["D4_money_paper_trail"]["usaspending_subaward_general"]
print(
    f"\nGeneral (STTR-population-wide) USASpending subaward extract present: "
    f"{subaward_general.exists()} ({subaward_general.relative_to(REPO_ROOT)})"
)
cohort_scoped = sorted((REPO_ROOT / "data" / "reports").glob("*/subaward*.csv")) + sorted(
    (REPO_ROOT / "data" / "reports").glob("*/*subawards.csv")
)
print(f"Cohort-scoped subaward extracts found instead: {[str(p.relative_to(REPO_ROOT)) for p in cohort_scoped]}")
print(
    "D4 subcontract-marker status: no population-wide extract; only technology-area-cohort-scoped "
    "extracts exist (e.g. nanotechnology), which do not cover the STTR population this cascade needs."
)

## D5 — text trail

Declared source: deterministic phrase lexicon over award abstracts and firm text. The lexicon
itself is [open question O-4](../../specs/sttr-spinout-linkage/open-questions.md) (not built here);
this only checks whether the abstract text D5 would run over is present and how complete it is.

In [ ]:
abstract_col = first_col(awards_raw, ("abstract", "Abstract")) if "awards_raw" in dir() else None
if abstract_col is None:
    print("D1 award frame not loaded (see D1 cell above) — cannot check D5 abstract coverage.")
else:
    abstract_coverage = sttr_p2[abstract_col].notna().sum()
    print(
        f"Abstract field non-null on STTR Phase II rows: {abstract_coverage:,} / {len(sttr_p2):,} "
        f"({abstract_coverage / len(sttr_p2):.1%})"
    )
    print(
        "Caveat carried from the Phase III DoD-description finding: overall non-null coverage can "
        "mask agency-level emptiness (short/boilerplate DoD abstracts). Not decomposed by agency here."
    )

## Sub-question 2 — partner-type seed-list capture status

Separate classifier, same D1 spine and freeze gate. `seed-list-provenance.md` already declares its
own status; this cell only surfaces that declaration next to the D1-D5 table instead of requiring a
second lookup.

In [ ]:
if not SEED_LIST_PROVENANCE.exists():
    print(f"Missing {SEED_LIST_PROVENANCE.relative_to(REPO_ROOT)}")
    seed_status = pd.DataFrame()
else:
    text = SEED_LIST_PROVENANCE.read_text(encoding="utf-8")
    rows = []
    for line in text.splitlines():
        if line.startswith("| `") and "|" in line[1:]:
            parts = [p.strip() for p in line.strip("|").split("|")]
            if len(parts) >= 6:
                rows.append(
                    {
                        "list": parts[0].strip("`"),
                        "purpose": parts[1],
                        "version": parts[3],
                        "captured": parts[4],
                    }
                )
    seed_status = pd.DataFrame(rows)
    n_pending = (seed_status["version"].str.contains("pending")).sum() if not seed_status.empty else 0
    print(f"{n_pending} / {len(seed_status)} seed lists still show 'version: _pending_' (not yet captured)")
seed_status

## Summary — dimension readiness

One row per dimension: whether *any* local artifact exists, and whether that artifact looks usable
for the STTR population (not just present).

In [ ]:
summary = pd.DataFrame(
    [
        {
            "dimension": "D1 award spine",
            "any_artifact_present": DIMENSION_INPUTS["D1_award_spine"]["awards"].exists(),
            "usable_for_sttr_population": True,
            "note": "STTR Phase II filter works today; RI/PI coverage checked above",
        },
        {
            "dimension": "D2 person trail",
            "any_artifact_present": False,
            "usable_for_sttr_population": False,
            "note": "ORCID live-API client only, no cache; no OpenAlex/PubMed client",
        },
        {
            "dimension": "D3 IP trail",
            "any_artifact_present": True,
            "usable_for_sttr_population": None,
            "note": "Bulk USPTO .csv.zip present and unprocessed; transformed jsonl looks synthetic",
        },
        {
            "dimension": "D4 money/paper trail",
            "any_artifact_present": True,
            "usable_for_sttr_population": None,
            "note": "Form D present (spinout marker); no general subaward extract (subcontract marker)",
        },
        {
            "dimension": "D5 text trail",
            "any_artifact_present": DIMENSION_INPUTS["D5_text_trail"]["awards"].exists(),
            "usable_for_sttr_population": True,
            "note": "Same source as D1; lexicon itself is O-4, not built here",
        },
        {
            "dimension": "Partner-type seed lists",
            "any_artifact_present": False,
            "usable_for_sttr_population": False,
            "note": "seed-list-provenance.md: all six lists still 'version: _pending_'",
        },
    ]
)
summary

## Findings and caveats

| Claim | Evidence/artifact | Caveat or alternative explanation |
|---|---|---|
| D1 (award spine) and D5 (text trail, same source) are ready today | `d1-check`, `d5-check` | RI/PI/Abstract coverage is aggregate, not decomposed by agency; DoD abstracts run thinner |
| D3 (IP trail) has real bulk source data but nothing processed yet | `d3-check` | The only "transformed" artifact found looks like a test fixture, not a real snapshot |
| D4 (money/paper trail) is half-covered | `d4-check` | Form D (spinout marker) exists; USASpending subaward (subcontract marker) has no population-wide extract |
| D2 (person trail) has no local data at all | `d2-check` | A live ORCID client exists but has never been run against the STTR population; OpenAlex/PubMed are unbuilt |
| Partner-type seed lists are entirely uncaptured | `seed-check` | `seed-list-provenance.md` already says this; confirmed against the file rather than assumed |

**So: data readiness is not what's blocking task 1.3.** Three of five cascade dimensions (D2, D3,
D4-subcontract-side) need real ingestion/processing work regardless of the `open-questions.md`
gate. Even if the gate cleared today, the cascade could not run end-to-end on current local
artifacts — D2 and the D4 subcontract marker would fall through to typed absence for the entire
population, which would make every award `INDETERMINATE` at Order 4 rather than exercise Orders 1-3
at all. Resolving `open-questions.md` is necessary but not sufficient for a first real run.


## Promotion checklist

- [x] Question and decision are explicit.
- [x] Data snapshot, population, grain, keys, and exclusions are recorded.
- [x] Samples and stochastic methods use a deterministic seed. *(N/A — no sampling here.)*
- [x] The destination tier and its contract are explicit (`exploratory`, non-citable).
- [ ] Recurring calculations have one canonical implementation.
- [ ] Recurring artifact generation has a thin CLI or Dagster asset.
- [ ] Citable work satisfies all four `evidence` contract requirements.
- [ ] Findings and methodology are linked from `docs/`.
- [x] Outputs and execution counts are cleared before commit.

Until promotion is complete, this notebook and its claims remain exploratory and non-citable. This
notebook does not itself move task 1.3 forward — it only tells you where to point ingestion effort
next, and confirms the `open-questions.md` gate is not the only blocker.
